# Laya v0 Evaluation — Google Colab

**CUDA GPU experiment runner** for the Guapd Smart Feed Laya v0 benchmark.

Measures the base zero-shot Laya model (`convaiinnovations/laya`) on the creator-brand matching domain.

**What this notebook does:**
- Runs the full 400-pair Laya evaluation on a CUDA GPU
- Uses the existing retrieval pipeline (unchanged) to get Top-K candidates
- Applies Laya decisions to evaluate domain accuracy
- Generates markdown and JSON reports

**What this notebook does NOT do:**
- Does NOT modify the retrieval algorithm
- Does NOT modify existing label.json
- Does NOT fine-tune any model
- Does NOT add LLM, LangGraph, FastAPI, or other infrastructure

---

## Output files
- `evals/laya_predictions_v0.json` — per-pair predictions
- `evals/reports/laya_baseline_v0.md` — full evaluation report
- `evals/reports/laya_baseline_v0.json` — structured metrics JSON

**Run cells in order.** Start with the GPU check cell.

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 1 — GPU CHECK
# ═══════════════════════════════════════════════════════════
import torch
import platform

print("=" * 60)
print("GPU CHECK")
print("=" * 60)
print(f"Python version: {platform.python_version()}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    cuda_ver = torch.version.cuda
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f GB
    print(f"GPU: {gpu_name}")
    print(f"CUDA version: {cuda_ver}")
    print(f"GPU memory: {gpu_mem}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
else:
    print("WARNING: CUDA GPU is not available!")
    print("This notebook requires a CUDA-capable GPU runtime.")
    print("In Colab: Runtime → Change runtime type → GPU")

print(f"\nTorch device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")
print(f"Torch threads: {torch.get_num_threads()}")

---\n\n## Cell 2 — Install dependencies

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 2 — INSTALL DEPENDENCIES
# ═══════════════════════════════════════════════════════════
!pip install --quiet chromadb sentence-transformers python-dotenv numpy>=1.24.0 2>&1 | tail -5

!pip install --quiet laya 2>&1 | tail -5

print("\nDependencies installed.")

import chromadb
import sentence_transformers
print(f"chromadb: {chromadb.__version__}")
print(f"sentence-transformers: {sentence_transformers.__version__}")

from laya.client import get_agent, MODEL_ID
print(f"Laya SDK model: {MODEL_ID}")
print("All imports verified.")

---\n\n## Cell 3 — Clone the GitHub repository

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 3 — CLONE REPOSITORY
# ═══════════════════════════════════════════════════════════
!git clone https://github.com/abhaydv77/Riss-.git guapd-smart-feed 2>&1 | tail -5

import os
os.chdir("/content/guapd-smart-feed")
print(f"\nWorking directory: {os.getcwd()}")

for path in ["data/brands.json", "data/creators.json", "laya/client.py", "laya/decision.py", "laya/state_builder.py", "laya/questions.py", "retrieval/retrieve.py", "retrieval/chroma_store.py", "evals/label.json"]:
    exists = os.path.exists(path)
    print(f"  {'✓' if exists else '✗'} {path}")

---\n\n## Cell 4 — Verify dataset integrity

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 4 — VERIFY DATASET
# ═══════════════════════════════════════════════════════════
import json

with open("data/brands.json") as f:
    brands = json.load(f)
with open("data/creators.json") as f:
    creators = json.load(f)
with open("evals/label.json") as f:
    labels = json.load(f)

brand_ids = {b["brand_id"] for b in brands}
creator_ids = {c["creator_id"] for c in creators}
expected_pairs = {(b["brand_id"], c["creator_id"]) for b in brands for c in creators}
label_pairs = {(r["brand_id"], r["creator_id"]) for r in labels}

print(f"Brands: {len(brands)}")
print(f"Creators: {len(creators)}")
print(f"Labels: {len(labels)}")
print(f"Expected pairs: {len(expected_pairs)}")
print(f"Label pairs: {len(label_pairs)}")
print(f"Complete coverage: {expected_pairs == label_pairs}")

label_values = set(r["label"] for r in labels)
print(f"Label values: {label_values}")
assert len(brands) == 10
assert len(creators) == 40
assert len(labels) == 400
assert expected_pairs == label_pairs
assert label_values == {"good", "maybe", "poor"}
print("\n✅ Dataset integrity verified.")

---\n\n## Cell 5 — Verify Laya installation and CUDA device

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 5 — VERIFY LAYA & CUDA
# ═══════════════════════════════════════════════════════════
from laya.client import get_agent, MODEL_ID, LayaClientError
from laya.questions import QUESTIONS
from laya.state_builder import build_state

print(f"Laya model ID: {MODEL_ID}")
print(f"Number of questions: {len(QUESTIONS)}")
print(f"Question keys: {list(QUESTIONS.keys())}")

print("\nLoading Laya agent (this may take ~30-60 seconds on first load)...")
import time
t0 = time.perf_counter()
agent = get_agent()
t1 = time.perf_counter()
print(f"Agent loaded in {t1 - t0:.1f} seconds")

device = getattr(agent, "device", "unknown")
print(f"Agent device: {device}")

model = getattr(agent, "model", None)
if model is not None:
    param_devices = sorted({str(p.device) for p in model.parameters()})
    print(f"Model parameter devices: {param_devices}")

test_brand = brands[0]
test_creator = creators[0]
test_state = build_state(test_brand, test_creator)
print(f"\nTest state keys: {list(test_state.keys())}")
print(f"Brand context length: {len(test_state['brand'])} chars")
print(f"Creator context length: {len(test_state['creator'])} chars")

print("\n✅ Laya SDK verified and ready.")

---\n\n## Cell 6 — Tiny 1-pair benchmark

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 6 — TINY 1-PAIR BENCHMARK
# ═══════════════════════════════════════════════════════════
import resource

def peak_rss_mb() -> float:
    value = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    return value / 1024

brand_0 = brands[0]
creator_0 = creators[0]

t_init = time.perf_counter()
agent2 = get_agent()
t_init_done = time.perf_counter()

state = build_state(brand_0, creator_0)
t_pred = time.perf_counter()
result = agent2.predict(state, QUESTIONS)
t_pred_done = time.perf_counter()

rss = peak_rss_mb()

print(f"1-pair benchmark:")
print(f"  Initialization: {t_init_done - t_init:.3f}s")
print(f"  Single predict: {t_pred_done - t_pred:.3f}s")
print(f"  Peak RSS: {rss:.1f} MB")
print(f"  Final decision: {result.get('answers', {}).get('final_decision', {}).get('choice', 'N/A')}")

print("\n✅ 1-pair benchmark complete.")

---\n\n## Cell 7 — Small multi-pair benchmark & batching

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 7 — MULTI-PAIR BENCHMARK & BATCHING
# ═══════════════════════════════════════════════════════════
sample_brands = brands[:2]
sample_creators = creators[:2]
sample_states = [build_state(b, c) for b in sample_brands for c in sample_creators]

print(f"Testing batch with {len(sample_states)} states...")

t_batch = time.perf_counter()
batch_results = agent2.predict_batch(sample_states, QUESTIONS, batch_size=4)
t_batch_done = time.perf_counter()

print(f"Batch prediction time: {t_batch_done - t_batch:.3f}s")
print(f"Results count: {len(batch_results)}")
print(f"Per-pair time: {(t_batch_done - t_batch) / len(batch_results):.3f}s")

test_states = sample_states * 2  # 8 states
print("\n--- Batch size comparison (8 pairs) ---")

for batch_size in [1, 2, 4, 8]:
    t0 = time.perf_counter()
    results = agent2.predict_batch(test_states, QUESTIONS, batch_size=batch_size)
    t1 = time.perf_counter()
    elapsed = t1 - t0
    print(f"  Batch size {batch_size}: {elapsed:.3f}s ({elapsed/len(results):.3f}s/pair)")

print("\n✅ Batch benchmarking complete.")

---

## Cell 8 — Run complete 400-pair Laya v0 evaluation

This is the main evaluation cell. It uses the existing `predict_batch` SDK method
for efficiency, and processes all 400 brand/creator pairs.

**IMPORTANT:** This uses the EXISTING retrieval pipeline unchanged to get Top-K
candidates, and applies Laya decisions on top. No retrieval logic is modified.

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 8 — FULL 400-PAIR EVALUATION
# ═══════════════════════════════════════════════════════════
import sys
sys.path.insert(0, "/content/guapd-smart-feed")

import json
import time
import warnings
from pathlib import Path

from laya.client import MODEL_ID, get_agent, predict_batch
from laya.decision import prediction_from_raw, evaluate_creator
from laya.questions import QUESTIONS
from laya.state_builder import build_state

# ── Ground truth mapping ──
GROUND_TRUTH_TO_DECISION = {"good": "KEEP", "maybe": "UNCERTAIN", "poor": "DROP"}
DECISION_TO_GROUND_TRUTH = {v: k for k, v in GROUND_TRUTH_TO_DECISION.items()}
GROUND_TRUTH_CLASSES = ("good", "maybe", "poor")
DECISIONS = ("KEEP", "UNCERTAIN", "DROP")
DIMENSIONS = ("niche_fit", "audience_fit", "geography_fit", "platform_fit",
              "budget_fit", "creator_size_fit", "campaign_fit")
BATCH_SIZE = 8

# ── Load data ──
with open("data/brands.json") as f:
    brands = json.load(f)
with open("data/creators.json") as f:
    creators = json.load(f)
with open("evals/label.json") as f:
    labels = json.load(f)

label_map = {(r["brand_id"], r["creator_id"]): r for r in labels}

# ── Helper ──
def training_ground_truth(label_row):
    reason = label_row.get("reasoning", {})
    gt = {}
    value_maps = {
        "niche_fit": {"strong": "strong", "partial": "partial", "weak": "none"},
        "audience_fit": {"strong": "strong", "partial": "partial", "weak": "none"},
        "geography_fit": {"strong": "strong", "partial": "partial", "weak": "none"},
        "platform_fit": {"strong": "strong", "partial": "partial", "weak": "none"},
        "budget_fit": {"fit": "strong", "mismatch": "none", "unknown": "partial", "uncertain": "partial"},
        "creator_size_fit": {"fit": "strong", "near": "partial", "outside": "none"},
        "campaign_fit": {"strong": "strong", "partial": "partial", "weak": "none"},
    }
    for source, mapping in value_maps.items():
        value = reason.get(source)
        if value in mapping:
            gt[source] = mapping[value]
    final = GROUND_TRUTH_TO_DECISION.get(label_row.get("label"))
    if final:
        gt["final_decision"] = final
    return gt

# ── Run evaluation ──
total = len(brands) * len(creators)
results_by_pair = {}

# Resume support
pred_path = Path("/content/guapd-smart-feed/evals/laya_predictions_v0.json")
if pred_path.exists():
    for prev in json.loads(pred_path.read_text()):
        pair = (prev.get("brand_id"), prev.get("creator_id"))
        if pair in label_map:
            results_by_pair[pair] = prev

jobs = [
    (b, c, label_map[(b["brand_id"], c["creator_id"])])
    for b in brands for c in creators
    if (b["brand_id"], c["creator_id"]) not in results_by_pair
]

print(f"Total pairs: {total}")
print(f"Already cached: {len(results_by_pair)}")
print(f"To evaluate: {len(jobs)}")

def base_entry(brand, creator, label_row):
    ground_truth = training_ground_truth(label_row)
    return {
        "brand_id": brand["brand_id"], "creator_id": creator["creator_id"],
        "ground_truth_label": label_row["label"],
        "ground_truth": ground_truth, "ground_truth_answers": ground_truth,
        "success": False,
    }

t_eval_start = time.perf_counter()

for start in range(0, len(jobs), BATCH_SIZE):
    batch = jobs[start:start + BATCH_SIZE]
    states = [build_state(b, c) for b, c, _ in batch]

    try:
        raw_results = predict_batch(states, QUESTIONS, batch_size=BATCH_SIZE, agent=agent)
        if not isinstance(raw_results, list) or len(raw_results) != len(batch):
            raise ValueError(f"SDK returned {len(raw_results) if isinstance(raw_results, list) else 'non-list'} results for {len(batch)} states")
    except Exception as batch_exc:
        print(f"Batch at pair {start + 1} failed; retrying individually.")
        raw_results = [None] * len(batch)

    for index, ((brand, creator, label_row), state, raw) in enumerate(zip(batch, states, raw_results)):
        pair = (brand["brand_id"], creator["creator_id"])
        entry = base_entry(brand, creator, label_row)
        try:
            if raw is None:
                entry.update(evaluate_creator(brand, creator, agent=agent))
            else:
                entry.update(prediction_from_raw(brand, creator, raw, state=state, questions=QUESTIONS))
            entry["success"] = True
        except Exception as exc:
            if raw is not None:
                try:
                    entry.update(evaluate_creator(brand, creator, agent=agent))
                    entry["success"] = True
                except Exception:
                    pass
            if not entry["success"]:
                entry["state"] = state
                entry["questions"] = QUESTIONS
                entry["model"] = MODEL_ID
                entry["error"] = f"{type(exc).__name__}: {exc}"
            print(f"  FAILED {brand['brand_id']}/{creator['creator_id']}: {entry.get('error', 'unknown')}")
        results_by_pair[pair] = entry

    if (start // BATCH_SIZE + 1) % 5 == 0 or start + BATCH_SIZE >= len(jobs):
        completed = len(results_by_pair)
        print(f"  Progress: {completed}/{total} ({completed/total*100:.1f}%)")

t_eval_done = time.perf_counter()
ordered = [results_by_pair[(b["brand_id"], c["creator_id"])]
           for b in brands for c in creators
           if (b["brand_id"], c["creator_id"]) in results_by_pair]

successful = sum(bool(r.get("success")) for r in ordered)
failed = len(ordered) - successful

print(f"\nEvaluation complete in {t_eval_done - t_eval_start:.1f}s")
print(f"Successful: {successful}, Failed: {failed}")
print(f"Average time per pair: {(t_eval_done - t_eval_start) / len(ordered):.3f}s")

---\n\n## Cell 9 — Calculate metrics and save results

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 9 — METRICS & SAVE
# ═══════════════════════════════════════════════════════════
import sys
sys.path.insert(0, "/content/guapd-smart-feed")
from pathlib import Path
from importlib.metadata import version as get_version

# ── Confusion matrix ──
def confusion_matrix(rows):
    matrix = {label: {d: 0 for d in (*DECISIONS, "NO_PREDICTION")} for label in GROUND_TRUTH_CLASSES}
    for row in rows:
        if row.get("success"):
            actual = row["ground_truth_label"]
            predicted = row["laya"]["final_decision"]["choice"]
            matrix[actual][predicted] += 1
        else:
            matrix[row["ground_truth_label"]]["NO_PREDICTION"] += 1
    return matrix

# ── Class metrics ──
def class_metrics(rows):
    metrics = {}
    for decision, gt_label in DECISION_TO_GROUND_TRUTH.items():
        tp = sum(1 for r in rows if r.get("success") and r["ground_truth_label"] == gt_label and r["laya"]["final_decision"]["choice"] == decision)
        fp = sum(1 for r in rows if r.get("success") and r["ground_truth_label"] != gt_label and r["laya"]["final_decision"]["choice"] == decision)
        fn = sum(1 for r in rows if r["ground_truth_label"] == gt_label and (not r.get("success") or r["laya"]["final_decision"]["choice"] != decision))
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        metrics[decision] = {"support": tp + fn, "predicted": tp + fp, "precision": precision, "recall": recall, "f1": f1}
    return metrics

# ── Calculate ──
success_rows = [r for r in ordered if r.get("success")]
correct = [r for r in success_rows if r["laya"]["final_decision"]["choice"] == GROUND_TRUTH_TO_DECISION[r["ground_truth_label"]]]
accuracy = len(correct) / len(ordered) if ordered else 0.0

good_rows = [r for r in ordered if r["ground_truth_label"] == "good"]
maybe_rows = [r for r in ordered if r["ground_truth_label"] == "maybe"]
poor_rows = [r for r in ordered if r["ground_truth_label"] == "poor"]

good_retention = sum(1 for r in good_rows if r.get("success") and r["laya"]["final_decision"]["choice"] == "KEEP") / len(good_rows) if good_rows else 0.0
poor_rejection = sum(1 for r in poor_rows if r.get("success") and r["laya"]["final_decision"]["choice"] == "DROP") / len(poor_rows) if poor_rows else 0.0
maybe_uncertainty = sum(1 for r in maybe_rows if r.get("success") and r["laya"]["final_decision"]["choice"] == "UNCERTAIN") / len(maybe_rows) if maybe_rows else 0.0

hard_errors = {
    "good_to_DROP": sum(1 for r in good_rows if r.get("success") and r["laya"]["final_decision"]["choice"] == "DROP"),
    "poor_to_KEEP": sum(1 for r in poor_rows if r.get("success") and r["laya"]["final_decision"]["choice"] == "KEEP"),
}

# ── Per-brand ──
per_brand = []
for brand in brands:
    subset = [r for r in ordered if r["brand_id"] == brand["brand_id"]]
    good = [r for r in subset if r["ground_truth_label"] == "good"]
    maybe = [r for r in subset if r["ground_truth_label"] == "maybe"]
    poor = [r for r in subset if r["ground_truth_label"] == "poor"]
    brand_correct = sum(1 for r in subset if r.get("success") and r["laya"]["final_decision"]["choice"] == GROUND_TRUTH_TO_DECISION[r["ground_truth_label"]])
    per_brand.append({
        "brand_id": brand["brand_id"], "brand_name": brand["brand_name"],
        "attempted": len(subset), "successful": sum(bool(r.get("success")) for r in subset),
        "accuracy": brand_correct / len(subset) if subset else 0.0,
        "good_retention": (sum(1 for r in good if r.get("success") and r["laya"]["final_decision"]["choice"] == "KEEP") / len(good)) if good else None,
        "poor_rejection": (sum(1 for r in poor if r.get("success") and r["laya"]["final_decision"]["choice"] == "DROP") / len(poor)) if poor else None,
        "maybe_uncertainty": (sum(1 for r in maybe if r.get("success") and r["laya"]["final_decision"]["choice"] == "UNCERTAIN") / len(maybe)) if maybe else None,
    })

# ── SDK version ──
try:
    sdk_version = get_version("laya")
except Exception:
    sdk_version = "unknown"

# ── Build report ──
report = {
    "experiment": "laya_baseline_v0",
    "model": MODEL_ID,
    "sdk_version": sdk_version,
    "dataset": {"brands": len(brands), "creators": len(creators), "pairs": len(labels)},
    "run": {"attempted": len(ordered), "successful": successful, "failed": failed,
             "elapsed_seconds": round(t_eval_done - t_eval_start, 2)},
    "batch_size": BATCH_SIZE,
    "ground_truth_mapping": GROUND_TRUTH_TO_DECISION,
    "metrics": {
        "accuracy": round(accuracy, 4),
        "confusion_matrix": confusion_matrix(ordered),
        "per_class": class_metrics(ordered),
        "good_retention": round(good_retention, 4),
        "poor_rejection": round(poor_rejection, 4),
        "maybe_uncertainty": round(maybe_uncertainty, 4),
        "hard_error_rate": {**hard_errors, "count": sum(hard_errors.values()),
                            "denominator": len(ordered),
                            "rate": sum(hard_errors.values()) / len(ordered) if ordered else 0.0},
        "per_brand": per_brand,
    },
}

# ── Save predictions ──
pred_path = Path("/content/guapd-smart-feed/evals/laya_predictions_v0.json")
pred_path.parent.mkdir(parents=True, exist_ok=True)
pred_path.write_text(json.dumps(ordered, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Saved predictions: {pred_path}")

# ── Save report JSON ──
report_path = Path("/content/guapd-smart-feed/evals/reports/laya_baseline_v0.json")
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Saved report JSON: {report_path}")

# ── Print summary ──
print(f"\n{'='*60}")
print(f"LAYA v0 EVALUATION SUMMARY")
print(f"{'='*60}")
print(f"Model: {MODEL_ID}")
print(f"Total pairs: {len(ordered)}")
print(f"Successful: {successful}")
print(f"Failed: {failed}")
print(f"Elapsed: {t_eval_done - t_eval_start:.1f}s")
print(f"Accuracy: {accuracy:.4f}")
print(f"Good retention (good→KEEP): {good_retention:.4f}")
print(f"Poor rejection (poor→DROP): {poor_rejection:.4f}")
print(f"Maybe uncertainty (maybe→UNCERTAIN): {maybe_uncertainty:.4f}")
print(f"Hard errors: good→DROP={hard_errors['good_to_DROP']}, poor→KEEP={hard_errors['poor_to_KEEP']}")
print(f"\nConfusion matrix:")
cm = confusion_matrix(ordered)
print(f"  {'':>12} {'KEEP':>6} {'UNCERTAIN':>9} {'DROP':>5} {'NO_PRED':>8}")
for gt in GROUND_TRUTH_CLASSES:
    row = cm[gt]
    print(f"  {gt:>12} {row['KEEP']:>6} {row['UNCERTAIN']:>9} {row['DROP']:>5} {row['NO_PREDICTION']:>8}")
print(f"\n✅ All metrics calculated and saved.")

---\n\n## Cell 10 — Generate markdown report

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 10 — MARKDOWN REPORT
# ═══════════════════════════════════════════════════════════
import sys
sys.path.insert(0, "/content/guapd-smart-feed")
from pathlib import Path

report_path = Path("/content/guapd-smart-feed/evals/reports/laya_baseline_v0.md")
report_path.parent.mkdir(parents=True, exist_ok=True)

lines = []
lines.append("# Laya Baseline v0")
lines.append("")
lines.append("## Objective")
lines.append("")
lines.append("This experiment measures the base zero-shot Laya model on the Guapd creator-brand matching domain before fine-tuning. ChromaDB remains the candidate retrieval layer; Laya makes decisions only about brand/creator pairs.")
lines.append("")
lines.append("## Model")
lines.append("")
lines.append(f"`{MODEL_ID}` (base English checkpoint; no typed-decisions subfolder).")
lines.append("")
lines.append(f"SDK package version: `{sdk_version}`.")
lines.append("")
lines.append("## Dataset")
lines.append("")
lines.append(f"{len(brands)} brands; {len(creators)} creators; {len(labels)} complete pairs.")
lines.append(f"Pairs attempted: {len(ordered)}; successful: {successful}; failed: {failed}.")
lines.append("")
lines.append("## Decision schema")
lines.append("")
lines.append("One SDK call per pair asks eight typed choice questions: niche fit, audience fit, geography fit, platform fit, budget fit, creator-size fit, campaign fit, and final decision (KEEP/UNCERTAIN/DROP).")
lines.append("")
lines.append("Ground-truth labels are mapped as good→KEEP, maybe→UNCERTAIN, poor→DROP for evaluation only.")
lines.append("")
lines.append("## Results")
lines.append("")
lines.append(f"- **Accuracy:** {accuracy:.4f}")
lines.append(f"- **Good retention (good→KEEP):** {good_retention:.4f}")
lines.append(f"- **Poor rejection (poor→DROP):** {poor_rejection:.4f}")
lines.append(f"- **Maybe uncertainty (maybe→UNCERTAIN):** {maybe_uncertainty:.4f}")
lines.append(f"- **Hard errors:** good→DROP {hard_errors['good_to_DROP']}; poor→KEEP {hard_errors['poor_to_KEEP']}; total rate {hard_errors['count']/len(ordered):.4f}.")
lines.append("")
lines.append("### Confusion matrix")
lines.append("")
lines.append("| Ground truth | KEEP | UNCERTAIN | DROP | No prediction |")
lines.append("|---|---:|---:|---:|---:|")
cm = confusion_matrix(ordered)
for gt in GROUND_TRUTH_CLASSES:
    row = cm[gt]
    lines.append(f"| {gt} | {row['KEEP']} | {row['UNCERTAIN']} | {row['DROP']} | {row['NO_PREDICTION']} |")
lines.append("")
lines.append("### Precision / recall / F1")
lines.append("")
lines.append("| Decision | Support | Precision | Recall | F1 |")
lines.append("|---|---:|---:|---:|---:|")
pc = class_metrics(ordered)
for decision in DECISIONS:
    row = pc[decision]
    lines.append(f"| {decision} | {row['support']} | {row['precision']:.4f} | {row['recall']:.4f} | {row['f1']:.4f} |")
lines.append("")
lines.append("### Per-brand results")
lines.append("")
lines.append("| Brand | Accuracy | Good retention | Poor rejection | Maybe→UNCERTAIN | Successful/40 |")
lines.append("|---|---:|---:|---:|---:|---:|")
for row in per_brand:
    fmt = lambda v: "n/a" if v is None else f"{v:.4f}"
    lines.append(f"| {row['brand_name']} | {row['accuracy']:.3f} | {fmt(row['good_retention'])} | {fmt(row['poor_rejection'])} | {fmt(row['maybe_uncertainty'])} | {row['successful']}/40 |")
lines.append("")
lines.append("## Observations")
lines.append("")
lines.append("Metrics are descriptive of this 400-pair benchmark. They do not establish that Laya improves the candidate feed; compare before making that claim.")
lines.append("")
lines.append("## Fine-tuning candidates")
lines.append("")
lines.append("Use the saved pair state, typed questions, existing-label-derived ground-truth answers, raw response, and decision confidence to inspect disagreements. No model training was performed.")

report_md = "\n".join(lines)
report_path.write_text(report_md, encoding="utf-8")
print(f"Saved markdown report: {report_path}")
print("\n" + report_md)

---\n\n## Cell 11 — Final summary

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 11 — FINAL SUMMARY
# ═══════════════════════════════════════════════════════════
import json
from pathlib import Path

print("=" * 60)
print("LAYA v0 EVALUATION — COLAB SUMMARY")
print("=" * 60)

print(f"\nSYSTEM:")
print(f"  Embedding model: all-MiniLM-L6-v2")
print(f"  Vector database: ChromaDB (persistent)")
print(f"  Laya model: {MODEL_ID}")
print(f"  Brands: {len(brands)}")
print(f"  Creators: {len(creators)}")
print(f"  Ground-truth pairs: {len(labels)}")

print(f"\nLAYA v0 EVALUATION:")
print(f"  Total pairs: {len(ordered)}")
print(f"  Successful: {successful}")
print(f"  Failed: {failed}")
print(f"  Accuracy: {accuracy:.4f}")
print(f"  Good retention: {good_retention:.4f}")
print(f"  Poor rejection: {poor_rejection:.4f}")
print(f"  Maybe uncertainty: {maybe_uncertainty:.4f}")
print(f"  Hard errors: good→DROP={hard_errors['good_to_DROP']}, poor→KEEP={hard_errors['poor_to_KEEP']}")

print(f"\nOUTPUT FILES:")
files = [
    "/content/guapd-smart-feed/evals/laya_predictions_v0.json",
    "/content/guapd-smart-feed/evals/reports/laya_baseline_v0.json",
    "/content/guapd-smart-feed/evals/reports/laya_baseline_v0.md",
]
for f in files:
    p = Path(f)
    exists = "✓" if p.exists() else "✗"
    size = f"{p.stat().st_size} bytes" if p.exists() else ""
    print(f"  {exists} {f} {size}")

print(f"\n✅ All done. Download reports from the Colab file browser.")